In [1]:
import os

print(os.listdir('/content'))


['.config', 'rating.csv', 'movie.csv', 'sample_data']


In [2]:
import pandas as pd
import os

for file in os.listdir('/content'):
    if file.endswith('.csv'):
        print("\nFile:", file)
        df = pd.read_csv('/content/' + file)
        print(df.head())
        print("Columns:", df.columns.tolist())


File: rating.csv
   userId  movieId  rating            timestamp
0       1        2     3.5  2005-04-02 23:53:47
1       1       29     3.5  2005-04-02 23:31:16
2       1       32     3.5  2005-04-02 23:33:39
3       1       47     3.5  2005-04-02 23:32:07
4       1       50     3.5  2005-04-02 23:29:40
Columns: ['userId', 'movieId', 'rating', 'timestamp']

File: movie.csv
   movieId                               title  \
0        1                    Toy Story (1995)   
1        2                      Jumanji (1995)   
2        3             Grumpier Old Men (1995)   
3        4            Waiting to Exhale (1995)   
4        5  Father of the Bride Part II (1995)   

                                        genres  
0  Adventure|Animation|Children|Comedy|Fantasy  
1                   Adventure|Children|Fantasy  
2                               Comedy|Romance  
3                         Comedy|Drama|Romance  
4                                       Comedy  
Columns: ['movieId', 'title'

In [3]:
import os

for file in os.listdir('/content'):
    if not file.startswith('.'):
        print("File name:", file)

File name: rating.csv
File name: movie.csv
File name: sample_data


In [4]:
import pandas as pd

# Load movie dataset
movies = pd.read_csv('/content/movie.csv')

# Load ratings dataset
ratings = pd.read_csv('/content/rating.csv')

# Display first 5 rows
print("MOVIE DATASET")
display(movies.head())

print("RATING DATASET")
display(ratings.head())

# Display dataset information
print("Movie dataset shape:", movies.shape)
print("Rating dataset shape:", ratings.shape)

print("Movie columns:", movies.columns.tolist())
print("Rating columns:", ratings.columns.tolist())

MOVIE DATASET


,movieId,title,genres
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
1,2,Jumanji (1995),Adventure|Children|Fantasy
2,3,Grumpier Old Men (1995),Comedy|Romance
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance
4,5,Father of the Bride Part II (1995),Comedy


RATING DATASET


,userId,movieId,rating,timestamp
0,1,2,3.5,2005-04-02 23:53:47
1,1,29,3.5,2005-04-02 23:31:16
2,1,32,3.5,2005-04-02 23:33:39
3,1,47,3.5,2005-04-02 23:32:07
4,1,50,3.5,2005-04-02 23:29:40


Movie dataset shape: (27278, 3)
Rating dataset shape: (823285, 4)
Movie columns: ['movieId', 'title', 'genres']
Rating columns: ['userId', 'movieId', 'rating', 'timestamp']


In [5]:
print("Movie Dataset Columns:")
print(movies.columns.tolist())

print("\nRating Dataset Columns:")
print(ratings.columns.tolist())

Movie Dataset Columns:
['movieId', 'title', 'genres']

Rating Dataset Columns:
['userId', 'movieId', 'rating', 'timestamp']


In [6]:
import pandas as pd

# Load datasets
movies = pd.read_csv('/content/movie.csv')
ratings = pd.read_csv('/content/rating.csv')

# Calculate average rating and total ratings for each movie
movie_stats = ratings.groupby('movieId').agg(
    average_rating=('rating', 'mean'),
    total_ratings=('rating', 'count')
).reset_index()

# Combine movie information with ratings
movie_data = movies.merge(movie_stats, on='movieId', how='inner')

# Show movies with the highest average ratings
# Consider only movies with at least 50 ratings
popular_movies = movie_data[movie_data['total_ratings'] >= 50].copy()

popular_movies = popular_movies.sort_values(
    by=['average_rating', 'total_ratings'],
    ascending=False
)

print("TOP 10 RECOMMENDED MOVIES")
display(
    popular_movies[
        ['title', 'genres', 'average_rating', 'total_ratings']
    ].head(10).round(2)
)

print("\nMovie Recommendation System is ready!")

TOP 10 RECOMMENDED MOVIES


,title,genres,average_rating,total_ratings
313,"Shawshank Redemption, The (1994)",Crime|Drama,4.47,3818
828,"Godfather, The (1972)",Crime|Drama,4.39,2545
49,"Usual Suspects, The (1995)",Crime|Mystery|Thriller,4.37,2901
7122,Band of Brothers (2001),Action|Drama|War,4.33,270
3273,Creature Comforts (1989),Animation|Comedy,4.31,144
12076,Cosmos (1980),Documentary,4.30,61
519,Schindler's List (1993),Drama|War,4.30,3101
2948,Matewan (1987),Drama,4.29,72
1175,"Godfather: Part II, The (1974)",Crime|Drama,4.29,1692
878,Casablanca (1942),Drama|Romance,4.28,1508



Movie Recommendation System is ready!


In [7]:
import pandas as pd
from IPython.display import display
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# Load movie dataset
movies = pd.read_csv('/content/movie.csv')

# Fill missing genres
movies['genres'] = movies['genres'].fillna('')

# Convert genres into numerical features
tfidf = TfidfVectorizer(token_pattern=r'[^|]+')
tfidf_matrix = tfidf.fit_transform(movies['genres'])

# Calculate similarity between movies
similarity_matrix = cosine_similarity(tfidf_matrix)

# Recommendation function
def recommend_movies(movie_name, number=10):
    matches = movies[
        movies['title'].str.contains(movie_name, case=False, na=False, regex=False)
    ]

    if matches.empty:
        print("Movie not found. Please try another movie name.")
        return

    movie_index = matches.index[0]

    similarity_scores = list(enumerate(similarity_matrix[movie_index]))
    similarity_scores = sorted(
        similarity_scores, key=lambda x: x[1], reverse=True
    )

    # Exclude the selected movie itself
    similar_movies = [
        (index, score)
        for index, score in similarity_scores
        if index != movie_index
    ][:number]

    print("Selected Movie:", movies.loc[movie_index, 'title'])
    print("\nRecommended Movies:")

    result = movies.iloc[[index for index, score in similar_movies]][
        ['title', 'genres']
    ].copy()

    result['similarity_score'] = [
        round(score, 3) for index, score in similar_movies
    ]

    display(result.reset_index(drop=True))

# Example
recommend_movies("Toy Story")

Selected Movie: Toy Story (1995)

Recommended Movies:


,title,genres,similarity_score
0,Antz (1998),Adventure|Animation|Children|Comedy|Fantasy,1.0
1,Toy Story 2 (1999),Adventure|Animation|Children|Comedy|Fantasy,1.0
2,"Adventures of Rocky and Bullwinkle, The (2000)",Adventure|Animation|Children|Comedy|Fantasy,1.0
3,"Emperor's New Groove, The (2000)",Adventure|Animation|Children|Comedy|Fantasy,1.0
4,"Monsters, Inc. (2001)",Adventure|Animation|Children|Comedy|Fantasy,1.0
5,DuckTales: The Movie - Treasure of the Lost La...,Adventure|Animation|Children|Comedy|Fantasy,1.0
6,"Wild, The (2006)",Adventure|Animation|Children|Comedy|Fantasy,1.0
7,Shrek the Third (2007),Adventure|Animation|Children|Comedy|Fantasy,1.0
8,"Tale of Despereaux, The (2008)",Adventure|Animation|Children|Comedy|Fantasy,1.0
9,Asterix and the Vikings (Astérix et les Viking...,Adventure|Animation|Children|Comedy|Fantasy,1.0


In [1]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from IPython.display import display

# Load movie dataset
movies = pd.read_csv('/content/movie.csv')
movies['genres'] = movies['genres'].fillna('').str.replace(
    '|', ' ', regex=False
)

# Convert genres into numerical features
tfidf = TfidfVectorizer()
tfidf_matrix = tfidf.fit_transform(movies['genres'])

# Calculate cosine similarity
similarity_matrix = cosine_similarity(tfidf_matrix)

def recommend_movies(movie_name, number=10):
    matches = movies[
        movies['title'].str.contains(
            movie_name, case=False, na=False, regex=False
        )
    ]

    if matches.empty:
        print("Movie not found. Try another title.")
        return

    print("Matching movies:")
    display(matches[['title', 'genres']].head(10))

    selected_index = matches.index[0]
    scores = list(enumerate(similarity_matrix[selected_index]))
    scores = sorted(scores, key=lambda x: x[1], reverse=True)

    recommendations = [
        (idx, score) for idx, score in scores
        if idx != selected_index
    ][:number]

    result = movies.loc[
        [idx for idx, score in recommendations], ['title', 'genres']
    ].copy()

    result['similarity_score'] = [
        round(score, 3) for idx, score in recommendations
    ]

    print("\nRecommended Movies:")
    display(result.reset_index(drop=True))

# Try another movie
recommend_movies("Jumanji")

Matching movies:


,title,genres
1,Jumanji (1995),Adventure Children Fantasy



Recommended Movies:


,title,genres,similarity_score
0,Kids of the Round Table (1995),Adventure Children Fantasy,1.0
1,"Indian in the Cupboard, The (1995)",Adventure Children Fantasy,1.0
2,"NeverEnding Story III, The (1994)",Adventure Children Fantasy,1.0
3,Escape to Witch Mountain (1975),Adventure Children Fantasy,1.0
4,Darby O'Gill and the Little People (1959),Adventure Children Fantasy,1.0
5,Return to Oz (1985),Adventure Children Fantasy,1.0
6,"NeverEnding Story, The (1984)",Adventure Children Fantasy,1.0
7,"NeverEnding Story II: The Next Chapter, The (1...",Adventure Children Fantasy,1.0
8,Santa Claus: The Movie (1985),Adventure Children Fantasy,1.0
9,Harry Potter and the Sorcerer's Stone (a.k.a. ...,Adventure Children Fantasy,1.0


In [2]:
!pip -q install gradio

In [4]:
import gradio as gr
import pandas as pd
from IPython.display import display

# Prepare movie titles for searching
movie_titles = sorted(movies['title'].dropna().unique().tolist())

def movie_recommender(selected_movie):
    if not selected_movie:
        return pd.DataFrame(columns=["Recommended Movie", "Genres", "Similarity Score"])

    matches = movies[movies['title'] == selected_movie]

    if matches.empty:
        return pd.DataFrame([{
            "Recommended Movie": "Movie not found",
            "Genres": "",
            "Similarity Score": ""
        }])

    movie_index = matches.index[0]

    scores = list(enumerate(similarity_matrix[movie_index]))
    scores = sorted(scores, key=lambda x: x[1], reverse=True)

    recommendations = [
        (idx, score) for idx, score in scores
        if idx != movie_index
    ][:10]

    results = []

    for idx, score in recommendations:
        results.append({
            "Recommended Movie": movies.loc[idx, "title"],
            "Genres": movies.loc[idx, "genres"],
            "Similarity Score": round(float(score), 3)
        })

    return pd.DataFrame(results)

demo = gr.Interface(
    fn=movie_recommender,
    inputs=gr.Dropdown(
        choices=movie_titles,
        label="Select a Movie",
        value=movie_titles[0] if movie_titles else None
    ),
    outputs=gr.Dataframe(label="Recommended Movies"),
    title="🎬 Movie Recommendation System",
    description="Select a movie to discover 10 similar movies."
)

demo.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()

Could not create share link. Please check your internet connection or our status page: https://status.gradio.app.


2026/10/29 13:13:05 [W] [service.go:132] login to server failed: dial tcp 44.237.78.176:7000: connect: connection refused


<IPython.core.display.Javascript object>

In [5]:
!pip install -q --upgrade gradio

In [8]:
# Test the recommendation system without Gradio

movie_name = "Toy Story"

matches = movies[
    movies["title"].str.contains(
        movie_name, case=False, na=False, regex=False
    )
]

if not matches.empty:
    selected_index = matches.index[0]

    scores = list(enumerate(similarity_matrix[selected_index]))
    scores = sorted(scores, key=lambda x: x[1], reverse=True)

    recommendations = [
        (idx, score)
        for idx, score in scores
        if idx != selected_index
    ][:10]

    result = movies.iloc[
        [idx for idx, score in recommendations]
    ][["title", "genres"]].copy()

    print("Selected Movie:", movies.loc[selected_index, "title"])
    print("Recommended Movies:")
    display(result.reset_index(drop=True))
else:
    print("Movie not found.")

Selected Movie: Toy Story (1995)
Recommended Movies:


,title,genres
0,Antz (1998),Adventure Animation Children Comedy Fantasy
1,Toy Story 2 (1999),Adventure Animation Children Comedy Fantasy
2,"Adventures of Rocky and Bullwinkle, The (2000)",Adventure Animation Children Comedy Fantasy
3,"Emperor's New Groove, The (2000)",Adventure Animation Children Comedy Fantasy
4,"Monsters, Inc. (2001)",Adventure Animation Children Comedy Fantasy
5,DuckTales: The Movie - Treasure of the Lost La...,Adventure Animation Children Comedy Fantasy
6,"Wild, The (2006)",Adventure Animation Children Comedy Fantasy
7,Shrek the Third (2007),Adventure Animation Children Comedy Fantasy
8,"Tale of Despereaux, The (2008)",Adventure Animation Children Comedy Fantasy
9,Asterix and the Vikings (Astérix et les Viking...,Adventure Animation Children Comedy Fantasy


In [10]:
import pandas as pd
import os

# Load both datasets
movies = pd.read_csv('/content/movie.csv')
ratings = pd.read_csv('/content/rating.csv')

print("Movies loaded:", movies.shape)
print("Ratings loaded:", ratings.shape)

print("Both datasets loaded successfully!")

Movies loaded: (27278, 3)
Ratings loaded: (4568405, 4)
Both datasets loaded successfully!


In [11]:
import os

project_folder = "/content/Movie_Recommendation_System"
os.makedirs(project_folder, exist_ok=True)

movies.to_csv(
    os.path.join(project_folder, "movie.csv"),
    index=False
)

ratings.to_csv(
    os.path.join(project_folder, "rating.csv"),
    index=False
)

print("Project folder created successfully!")
print(os.listdir(project_folder))

Project folder created successfully!
['rating.csv', 'movie.csv']


In [12]:
import os

project_folder = "/content/Movie_Recommendation_System"


readme_text = """

"""

with open(os.path.join(project_folder, "README.md"), "w") as f:
    f.write(readme_text)

print("README.md created successfully!")
print(os.listdir(project_folder))

README.md created successfully!
['rating.csv', 'README.md', 'movie.csv']


In [13]:
import shutil

zip_path = shutil.make_archive(
    "/content/Movie_Recommendation_System",
    "zip",
    "/content/Movie_Recommendation_System"
)

print("ZIP file created:", zip_path)

ZIP file created: /content/Movie_Recommendation_System.zip


In [14]:
from google.colab import files

files.download("/content/Movie_Recommendation_System.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>